# Telugu FastSpeech 2: Notebook 1 (data audit)

**Runtime: CPU only** (Runtime > Change runtime type > CPU). No GPU is needed until training.

Dataset: `SPRINGLab/IndicTTS_Telugu` (derived from the IIT Madras Indic TTS database; read and respect its licence).
All results are saved to Google Drive under `tts_project/`, so a disconnect loses nothing.

Run the cells one at a time and paste the outputs back to Claude.

In [16]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q datasets soundfile
import io, os, collections, soundfile as sf, pandas as pd
from datasets import load_dataset, Audio
os.makedirs("/content/drive/MyDrive/tts_project", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Step 1: load the dataset (about 6.4 GB; takes several minutes)
ds = load_dataset("SPRINGLab/IndicTTS_Telugu", split="train")
print(ds)
print(ds.column_names)
acol = [k for k, v in ds.features.items() if v.__class__.__name__ == "Audio"][0]
print("audio column:", acol, "|", {k: v for k, v in ds[0].items() if k != acol})

README.md:   0%|          | 0.00/3.25k [00:00<?, ?B/s]

data/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B / 1.20GB            

data/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00004.parquet: reconstructing file:   0%|          |  0.00B / 1.36GB            

data/train-00001-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00004.parquet: reconstructing file:   0%|          |  0.00B / 2.15GB            

data/train-00002-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00003-of-00004.parquet: reconstructing file:   0%|          |  0.00B / 1.73GB            

data/train-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8576 [00:00<?, ? examples/s]

Dataset({
    features: ['audio', 'text', 'gender'],
    num_rows: 8576
})
['audio', 'text', 'gender']
audio column: audio | {'text': 'నరకవాసి.', 'gender': 0}


In [ ]:
# Step 2: audit every clip without decoding the audio
raw = ds.cast_column(acol, Audio(decode=False))
rows = []
for i, ex in enumerate(raw):
    a = ex[acol]; info = sf.info(io.BytesIO(a["bytes"]))
    rows.append(dict(idx=i, path=a["path"], sr=info.samplerate, ch=info.channels, dur_s=info.duration,
                     **{k: ex[k] for k in raw.column_names if k != acol}))
df = pd.DataFrame(rows)
print(len(df), "clips |", round(df.dur_s.sum()/3600, 2), "hours | sr:", df.sr.value_counts().to_dict(), "| ch:", df.ch.value_counts().to_dict())
print(df.dur_s.describe().round(2))

tcol = next(c for c in df.columns if df[c].dtype == object and df[c].astype(str).str.contains(r"[\u0C00-\u0C7F]").any())
print("text column:", tcol)
for c in df.columns:
    if c not in (tcol, "path", "idx") and df[c].nunique() < 20:
        print(c, "->", (df.groupby(c).dur_s.sum()/3600).round(2).to_dict())
print(df.path.head(4).tolist()); print(df[tcol].head(4).tolist())
chars = collections.Counter("".join(df[tcol]))
print("non-Telugu characters:", {k: v for k, v in chars.items() if not ("\u0C00" <= k <= "\u0C7F")})
print("distinct Telugu code points:", sum(1 for k in chars if "\u0C00" <= k <= "\u0C7F"))
print("clips with digits:", df[tcol].str.contains(r"\d").sum(), "| with Latin letters:", df[tcol].str.contains(r"[A-Za-z]").sum())

8576 clips | 36.71 hours | sr: {48000: 4533, 16000: 4043} | ch: {1: 8576}
count    8576.00
mean       15.41
std        12.44
min         0.66
25%         7.03
50%        10.97
75%        20.55
max       151.63
Name: dur_s, dtype: float64
text column: text
sr -> {16000: 23.92, 48000: 12.79}
ch -> {1: 36.71}
gender -> {0: 23.92, 1: 12.79}
['train_telugufemale_00001.wav', 'train_telugufemale_00002.wav', 'train_telugufemale_00003.wav', 'train_telugufemale_00004.wav']
['నరకవాసి.', 'బ్రహ్మదత్తుడు కాశీరాజ్యాన్ని పరిపాలించే కాలంలో ఆ నగరంలో ధనికుడైన ఒక గొప్పవర్తకుడుండేవాడు ఆయనకు మిత్ర విందకుడని ఒక కొడుకుండే వాడు ఈ మిత్రవిందకుడు ఎంతో పాపాత్ముడు వర్తకుడు చనిపోయాక ఆయన భార్య తన కుమారుడికి నాయనా దానాలు చెయ్యి నియమాలు పాటించు ధర్మం అనుసరించు అని ఎంతో హితబోధ చేసింది కాని వాడు తల్లి మాటలు కొంచెమైనా వినిపించుకోలేదు.', 'ఇంతలో కార్తీక పౌర్ణమి వచ్చింది మిత్రవిందకుడితో తల్లి నాయనా ఇవాళ పుణ్యదినం తెల్లవార్లూ విహారంలో ధర్మోపదేశం చేస్తారు నీవు అక్కడ అందరితోపాటు పూజచేసుకుని ఉపదేశం వినిరా తిరిగివచ్చాక నీకు వెయ్య

In [ ]:
# Step 3: save the audit to Drive
df.to_csv("/content/drive/MyDrive/tts_project/telugu_audit.csv", index=False)
print("saved", len(df), "rows")

saved 8576 rows


**Paste back:** the column names and first sample (step 1), and the printed summary from step 2
(hours, sampling rate, duration stats, hours per speaker or gender, non-Telugu characters, digit and Latin counts).

In [ ]:
import pandas as pd, numpy as np, collections, unicodedata, re
df = pd.read_csv("/content/drive/MyDrive/tts_project/telugu_audit.csv")
df["who"] = df.path.str.extract(r"telugu(female|male)")[0]
df["text"] = df.text.astype(str)

print(pd.crosstab([df.gender, df.who], df.sr))
bins = [0, 2, 4, 6, 8, 10, 12, 15, 20, 30, 200]
df["band"] = pd.cut(df.dur_s, bins)
t = df.groupby(["who", "band"], observed=True).dur_s.agg(clips="size", hours=lambda s: round(s.sum()/3600, 2)).unstack(0)
print(t)

df["cps"] = df.text.str.len() / df.dur_s
print(df.groupby("who").cps.describe().round(1))
print("duplicate texts:", df.text.str.strip().duplicated().sum())

print("--- Latin 'o' contexts ---")
for s in df.text[df.text.str.contains("o")].head(6): print(repr(s[:90]))
print("BOM clips:", df.text.str.contains("\ufeff").sum())

print("--- Telugu code point inventory ---")
chars = collections.Counter("".join(df.text))
for ch, n in sorted(chars.items(), key=lambda kv: -kv[1]):
    if "\u0C00" <= ch <= "\u0C7F":
        print(f"U+{ord(ch):04X} {unicodedata.name(ch, '?'):38s} {n}")

sr             16000  48000
gender who                 
0      female   4043      0
1      male        0   4533
           clips       hours      
who       female male female  male
band                              
(0, 2]        73   66   0.03  0.03
(2, 4]       118  447   0.09  0.43
(4, 6]       198  678   0.28  0.93
(6, 8]       254  903   0.49  1.77
(8, 10]      241  884   0.61  2.19
(10, 12]     265  500   0.81  1.51
(12, 15]     390  338   1.46  1.25
(15, 20]     685  300   3.32  1.44
(20, 30]     917  307   6.26  2.05
(30, 200]    902  110  10.57  1.19
         count  mean  std  min   25%   50%   75%   max
who                                                   
female  4043.0   9.2  1.6  0.4   8.8   9.2   9.7  39.7
male    4533.0  12.6  2.2  3.5  11.4  12.8  13.9  84.9
duplicate texts: 113
--- Latin 'o' contexts ---
'జమిందారుకు గంగన్నమీద గురి కుదిరింది, వద్దన్నా బలవంతాన అతనికి చాలా ధనం ఇచ్చాడు వీళుoటే, మరో'
'రాజకుమారుడు ఆ స్త్రీని సమీపించి, ఏమమ్మా ఎవరు నీవు, ఇది ఏ నగరం, నీ పేరేo

In [ ]:
import pandas as pd, numpy as np, re, random, collections
D = "/content/drive/MyDrive/tts_project"
df = pd.read_csv(f"{D}/telugu_audit.csv")
df["who"] = df.path.str.extract(r"telugu(female|male)")[0]
df["spk"] = df.who.map({"female": "TEF", "male": "TEM"})

def clean(t):
    t = str(t).replace("\ufeff", "").replace("o", "\u0C02")
    t = re.sub(r"[.,;:!?\"'\u201d\u201c\u2019\u2018]", " ", t)
    return re.sub(r"\s+", " ", t).strip()
df["norm"] = df.text.map(clean)
bad = df.norm.map(lambda s: any(not (ch == " " or "\u0C00" <= ch <= "\u0C7F") for ch in s))
print("clips with non-Telugu chars after cleaning:", int(bad.sum())); df = df[~bad]

n0 = len(df)
df = df[df.dur_s.between(2, 15)].copy()
df["cps"] = df.norm.str.len() / df.dur_s
z = df.groupby("spk").cps.transform(lambda s: (s - s.mean()) / s.std())
print("after 2-15 s:", len(df), "of", n0, "| dropped by chars/sec outlier rule:", int((z.abs() > 3).sum()))
df = df[z.abs() <= 3].copy()
print((df.groupby("spk").dur_s.sum() / 3600).round(2).to_dict(), "hours available per speaker")
print("texts shared by both speakers:", int((df.groupby("norm").spk.nunique() > 1).sum()))

random.seed(42)
VAL_T, TR_T = 0.5 * 3600, 2.5 * 3600
groups = [g for _, g in df.groupby("norm")]; random.shuffle(groups)

# pass 1: validation groups (a text belongs to exactly one split)
val_s = collections.Counter(); val_parts = []; val_texts = set()
for g in groups:
    per = g.groupby("spk").dur_s.sum()
    if all(val_s[s] < VAL_T for s in per.index):
        for s, d in per.items(): val_s[s] += d
        val_parts.append(g.assign(split="val")); val_texts.add(g.norm.iloc[0])

# pass 2: training rows from the remaining texts, capped per speaker
tr_s = collections.Counter(); tr_parts = []
for g in groups:
    if g.norm.iloc[0] in val_texts: continue
    keep = []
    for r in g.itertuples():
        if tr_s[r.spk] < TR_T:
            tr_s[r.spk] += r.dur_s; keep.append(r.Index)
    if keep: tr_parts.append(g.loc[keep].assign(split="train"))

sel = pd.concat(val_parts + tr_parts, ignore_index=True)
sel["utt"] = sel.spk + "_" + sel.idx.astype(str).str.zfill(5)

tr, va = sel[sel.split == "train"], sel[sel.split == "val"]
assert not (set(tr.norm) & set(va.norm)), "text overlap"
assert not (set(tr.utt) & set(va.utt)), "clip overlap"
assert set(va.spk) <= set(tr.spk)
print(f"train: {len(tr)} clips, {tr.dur_s.sum()/3600:.2f} h | val: {len(va)} clips, {va.dur_s.sum()/3600:.2f} h")
print({k: round(float(v), 2) for k, v in (sel.groupby(["split", "spk"]).dur_s.sum() / 3600).items()})
print("max clip s:", round(float(sel.dur_s.max()), 2), "| remaining 'o' in text:", int(sel.norm.str.contains("o").sum()))
print(sel.norm.head(3).tolist())
sel.to_csv(f"{D}/telugu_split.csv", index=False)
print("saved", len(sel), "rows")

clips with non-Telugu chars after cleaning: 0
after 2-15 s: 5216 of 8576 | dropped by chars/sec outlier rule: 26
{'TEF': 3.7, 'TEM': 8.07} hours available per speaker
texts shared by both speakers: 562
train: 2147 clips, 5.00 h | val: 419 clips, 1.00 h
{('train', 'TEF'): 2.5, ('train', 'TEM'): 2.5, ('val', 'TEF'): 0.5, ('val', 'TEM'): 0.5}
max clip s: 14.97 | remaining 'o' in text: 0
['ఈమె పశ్చిమ బెంగాల్ నుండి తెలుగు పరిశ్రమకు వచ్చినది', 'ఇది బాపట్ల ఇంజనీరింగ్ కళాశాలలో భాగం కాదు', 'ఈ కార్యక్రమం మూడేళ్ళ పాటు కొనసాగుతుందని ఆరోగ్య శాఖ మంత్రి డాక్టర్ యోగానంద శాస్త్రి అన్నారు']
saved 2566 rows


In [ ]:
import io, os, tarfile, numpy as np, soundfile as sf, pandas as pd
from scipy.signal import resample_poly
from datasets import load_dataset, Audio
D = "/content/drive/MyDrive/tts_project"
sel = pd.read_csv(f"{D}/telugu_split.csv")
try: raw
except NameError:
    ds = load_dataset("SPRINGLab/IndicTTS_Telugu", split="train")
    acol = [k for k, v in ds.features.items() if v.__class__.__name__ == "Audio"][0]
    raw = ds.cast_column(acol, Audio(decode=False))
acol = [k for k, v in raw.features.items() if v.__class__.__name__ == "Audio"][0]

RATIO = {16000: (441, 320), 48000: (147, 320)}      # to 22,050 Hz
root = "/content/raw_data/Telugu"
total = 0.0
for r in sel.itertuples():
    wav, sr = sf.read(io.BytesIO(raw[int(r.idx)][acol]["bytes"]), dtype="float32")
    if wav.ndim > 1: wav = wav.mean(1)
    up, down = RATIO[sr]
    wav = resample_poly(wav, up, down)
    wav = wav / max(1e-6, np.abs(wav).max()) * 0.95
    d = f"{root}/{r.spk}"; os.makedirs(d, exist_ok=True)
    sf.write(f"{d}/{r.utt}.wav", wav, 22050, subtype="PCM_16")
    open(f"{d}/{r.utt}.lab", "w", encoding="utf-8").write(r.norm)
    total += len(wav) / 22050
with tarfile.open(f"{D}/raw_data_telugu.tar", "w") as tf: tf.add("/content/raw_data", arcname="raw_data")
print("exported", len(sel), "utterances |", round(total / 3600, 2), "hours | speakers:",
      sorted(os.listdir(root)))

exported 2566 utterances | 6.01 hours | speakers: ['TEF', 'TEM']


In [ ]:
import re, os, shutil, collections, pandas as pd
D = "/content/drive/MyDrive/tts_project"
IV = {0x0C05:"A",0x0C06:"AA",0x0C07:"I",0x0C08:"II",0x0C09:"U",0x0C0A:"UU",0x0C0B:"R U",0x0C60:"R UU",
      0x0C0E:"E",0x0C0F:"EE",0x0C10:"AI",0x0C12:"O",0x0C13:"OO",0x0C14:"AU"}
VS = {0x0C3E:"AA",0x0C3F:"I",0x0C40:"II",0x0C41:"U",0x0C42:"UU",0x0C43:"R U",0x0C44:"R UU",
      0x0C46:"E",0x0C47:"EE",0x0C48:"AI",0x0C4A:"O",0x0C4B:"OO",0x0C4C:"AU"}
CONS = {0x0C15:"K",0x0C16:"KH",0x0C17:"G",0x0C18:"GH",0x0C19:"NG",0x0C1A:"C",0x0C1B:"CH",0x0C1C:"J",
        0x0C1D:"JH",0x0C1E:"NY",0x0C1F:"TT",0x0C20:"TTH",0x0C21:"DD",0x0C22:"DDH",0x0C23:"NN",
        0x0C24:"T",0x0C25:"TH",0x0C26:"D",0x0C27:"DH",0x0C28:"N",0x0C2A:"P",0x0C2B:"PH",0x0C2C:"B",
        0x0C2D:"BH",0x0C2E:"M",0x0C2F:"Y",0x0C30:"R",0x0C31:"RR",0x0C32:"L",0x0C33:"LL",0x0C35:"V",
        0x0C36:"SH",0x0C37:"SS",0x0C38:"S",0x0C39:"H"}
VIRAMA, ANUSVARA, VISARGA = 0x0C4D, 0x0C02, 0x0C03
NASAL = {**dict.fromkeys(["P","PH","B","BH","M"], "M"), **dict.fromkeys(["K","KH","G","GH"], "NG"),
         **dict.fromkeys(["C","CH","J","JH"], "NY"), **dict.fromkeys(["TT","TTH","DD","DDH","NN"], "NN"),
         **dict.fromkeys(["T","TH","D","DH","N"], "N")}
def g2p_word(w):
    out, i, n = [], 0, len(w)
    while i < n:
        c = ord(w[i])
        if c in IV: out += IV[c].split(); i += 1
        elif c in CONS:
            out.append(CONS[c]); i += 1
            if i < n and ord(w[i]) == VIRAMA: i += 1
            elif i < n and ord(w[i]) in VS: out += VS[ord(w[i])].split(); i += 1
            else: out.append("A")
        elif c == ANUSVARA:
            nxt = CONS.get(ord(w[i+1])) if i + 1 < n else None
            out.append(NASAL.get(nxt, "M")); i += 1
        elif c == VISARGA: out.append("H"); i += 1
        else: raise ValueError(f"unhandled character U+{c:04X} in {w!r}")
    return out

sel = pd.read_csv(f"{D}/telugu_split.csv")
words = sorted({w for t in sel.norm for w in t.split()})
wid = {w: f"w{i:05d}" for i, w in enumerate(words)}
phones, failed = collections.Counter(), []
os.makedirs("/content/mfa_telugu", exist_ok=True)
with open("/content/mfa_telugu/dict_ids.txt", "w") as f, open("/content/mfa_telugu/word_ids.tsv", "w", encoding="utf-8") as m:
    for w in words:
        try:
            p = g2p_word(w); phones.update(p)
            f.write(f"{wid[w]}\t{' '.join(p)}\n"); m.write(f"{wid[w]}\t{w}\t{' '.join(p)}\n")
        except ValueError as e: failed.append(str(e))
print("words:", len(words), "| failed:", len(failed), failed[:5])
print("phones used (%d):" % len(phones), dict(phones.most_common()))

src, dst = "/content/raw_data/Telugu", "/content/raw_aligner/Telugu"
shutil.rmtree(dst, ignore_errors=True)
n = 0
for spk in os.listdir(src):
    os.makedirs(f"{dst}/{spk}", exist_ok=True)
    for fn in os.listdir(f"{src}/{spk}"):
        if fn.endswith(".wav"): os.link(f"{src}/{spk}/{fn}", f"{dst}/{spk}/{fn}")
        elif fn.endswith(".lab"):
            t = open(f"{src}/{spk}/{fn}", encoding="utf-8").read().split()
            open(f"{dst}/{spk}/{fn}", "w").write(" ".join(wid[w] for w in t if w in wid)); n += 1
print("aligner labels written:", n)
os.system(f"cp /content/mfa_telugu/dict_ids.txt /content/mfa_telugu/word_ids.tsv {D}/")

words: 12127 | failed: 0 []
phones used (47): {'A': 15500, 'I': 8306, 'U': 8097, 'AA': 7373, 'N': 6995, 'R': 5322, 'K': 4649, 'L': 4335, 'M': 3806, 'T': 3426, 'P': 3285, 'V': 3166, 'D': 2945, 'DD': 2512, 'EE': 2487, 'C': 2451, 'S': 2347, 'TT': 2193, 'Y': 2113, 'G': 2100, 'OO': 1928, 'E': 1486, 'NN': 1209, 'II': 1121, 'UU': 967, 'NY': 820, 'B': 787, 'J': 715, 'O': 676, 'SH': 604, 'H': 498, 'NG': 485, 'AI': 483, 'SS': 474, 'DH': 470, 'LL': 451, 'BH': 385, 'KH': 140, 'TH': 121, 'AU': 68, 'PH': 68, 'GH': 57, 'TTH': 46, 'CH': 19, 'DDH': 15, 'RR': 6, 'JH': 3}
aligner labels written: 2566


0

In [ ]:
import os, subprocess
D = "/content/drive/MyDrive/tts_project"
if not os.path.exists("/content/mfa_env"):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
    !/content/bin/micromamba create -y -q -p /content/mfa_env -c conda-forge python=3.11 montreal-forced-aligner
!MFA_ROOT_DIR=/content/mfa_root /content/bin/micromamba run -p /content/mfa_env mfa version
os.makedirs("/content/mfa_logs", exist_ok=True)
nj = os.cpu_count() or 2
cmd = (f"MFA_ROOT_DIR=/content/mfa_root /content/bin/micromamba run -p /content/mfa_env "
       f"mfa train --clean -j {nj} --output_directory /content/mfa_telugu/TextGrid "
       f"/content/raw_aligner/Telugu /content/mfa_telugu/dict_ids.txt /content/mfa_telugu/telugu_acoustic.zip "
       f"> /content/mfa_logs/train.log 2>&1 "
       f"&& cd /content/mfa_telugu && tar -cf {D}/textgrids_telugu.tar TextGrid "
       f"&& cp telugu_acoustic.zip {D}/ && echo ALIGN_DONE_AND_SAVED >> /content/mfa_logs/train.log")
subprocess.Popen(["bash", "-c", cmd], start_new_session=True)
print(f"Telugu aligner started with {nj} CPU jobs")

3.4.3
Telugu aligner started with 2 CPU jobs


In [ ]:
b = subprocess.run("tr '\\r' '\\n' < /content/mfa_logs/train.log | grep -a -E 'INFO|ERROR|WARNING|Error' | tail -6", shell=True, capture_output=True).stdout
print(b.decode("utf-8", errors="replace"))

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
 INFO     Found 2 speakers across 2566 files, average number of utterances per  
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 INFO     Generating MFCCs...                                                   



In [ ]:
D = "/content/drive/MyDrive/tts_project"
import os, shutil, yaml, pandas as pd
!pip install -q pyworld pypinyin unidecode inflect librosa pyyaml tgt
if not os.path.isdir("/content/FastSpeech2"):
    !git clone -q https://github.com/ming024/FastSpeech2.git /content/FastSpeech2
    !cd /content/FastSpeech2 && git apply {D}/repo_patches.diff && echo "patches applied"
    !cd /content/FastSpeech2/hifigan && unzip -o -q generator_universal.pth.tar.zip
os.chdir("/content/FastSpeech2")

sel = pd.read_csv(f"{D}/telugu_split.csv")
open("split_val_utts.txt", "w").write("\n".join(sel.utt[sel.split == "val"]))
print("val utts:", (sel.split == "val").sum(), "| train utts:", (sel.split == "train").sum())

shutil.copytree("config/LJSpeech", "config/Telugu", dirs_exist_ok=True)
def edit(name, fn):
    p = f"config/Telugu/{name}.yaml"
    c = yaml.safe_load(open(p)); fn(c); yaml.safe_dump(c, open(p, "w"), sort_keys=False)
def pre(c):
    c["dataset"] = "Telugu"
    c["path"].update(corpus_path="/content/raw_data/Telugu", raw_path="/content/raw_data/Telugu",
                     preprocessed_path="/content/FastSpeech2/preprocessed_data/Telugu")
def mod(c):
    c["multi_speaker"] = True; c["vocoder"]["speaker"] = "universal"; c["max_seq_len"] = 1500
def tr(c):
    c["path"].update(ckpt_path=f"{D}/output/ckpt/Telugu", log_path=f"{D}/output/log/Telugu",
                     result_path=f"{D}/output/result/Telugu")
    c["step"].update(total_step=30000, log_step=100, synth_step=2000, val_step=1000, save_step=2000)
edit("preprocess", pre); edit("model", mod); edit("train", tr)
!mkdir -p {D}/config_Telugu && cp config/Telugu/*.yaml {D}/config_Telugu/
!grep -nE "multi_speaker|speaker:|max_seq_len|ckpt_path|total_step|save_step" config/Telugu/model.yaml config/Telugu/train.yaml
print("---------- text/symbols.py ----------")
!cat text/symbols.py

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
patches applied
val utts: 419 | train utts: 2147
config/Telugu/model.yaml:22:multi_speaker: true
config/Telugu/model.yaml:23:max_seq_len: 1500
config/Telugu/model.yaml:26:  speaker: universal
config/Telugu/train.yaml:2:  ckpt_path: /content/drive/MyDrive/tts_project/output/ckpt/Telugu
config/Telugu/train.yaml:21:  total_step: 30000
config/Telugu/train.yaml:25:  save_step: 2000
---------- text/symbols.py ----------
""" from https://github.com/keithito/tacotron """

"""
Defines the set of symbols used in text input to the model.

The default is a set of ASCII characters that works well for English or text that has been run through Unidecode. For other data, you can modify _characters. See TRAINING_DATA.md for details. """

from text import cmudict, pinyin

_pad = "_"
_punctuation = "!'(),.:;? "
_special = "-"
_letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefgh

In [17]:
import subprocess
running = subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True).returncode == 0
b = subprocess.run("tr '\\r' '\\n' < /content/mfa_logs/train.log | grep -a -E 'INFO|ERROR|ALIGN_DONE' | tail -3", shell=True, capture_output=True).stdout
print("TELUGU ALIGNER:", "RUNNING" if running else "NOT RUNNING"); print(b.decode("utf-8", errors="replace"))

TELUGU ALIGNER: RUNNING
 INFO     Accumulating statistics...                                            
 INFO     monophone - Iteration 4 of 40                                         
 INFO     Generating alignments...                                              



In [18]:
import os, sys, importlib
os.chdir("/content/FastSpeech2")
D = "/content/drive/MyDrive/tts_project"
p = "text/symbols.py"
s = open(p).read()
TELUGU = ("A AA I II U UU E EE AI O OO AU "
          "K KH G GH NG C CH J JH NY TT TTH DD DDH NN T TH D DH N P PH B BH M Y R RR L LL V SH SS S H").split()
old_a = '_arpabet = ["@" + s for s in cmudict.valid_symbols]'
old_p = '_pinyin = ["@" + s for s in pinyin.valid_symbols]'
print("arpabet line found:", s.count(old_a), "| pinyin line found:", s.count(old_p))
if s.count(old_a) == 1 and s.count(old_p) == 1:
    s = s.replace(old_a, "_arpabet = " + repr(["@" + t for t in TELUGU]) + "  # Telugu phone set (replaces ARPAbet)")
    s = s.replace(old_p, "_pinyin = []")
    open(p, "w").write(s)
    print("patched")

for m in [k for k in sys.modules if k == "text" or k.startswith("text.")]: del sys.modules[m]
sys.path.insert(0, "/content/FastSpeech2")
from text import text_to_sequence, sequence_to_text
from text.symbols import symbols
print("total symbols:", len(symbols))

# every phone in our dictionary must exist as a symbol
used = {ph for l in open("/content/mfa_telugu/word_ids.tsv", encoding="utf-8") for ph in l.rstrip("\n").split("\t")[2].split()}
print("dictionary phones:", len(used), "| missing from symbols:", sorted(x for x in used if "@" + x not in symbols))
seq = text_to_sequence("{A DD A V I L OO}", ["english_cleaners"])
print(seq, "->", sequence_to_text(seq))
!git diff text/symbols.py > {D}/telugu_symbols.diff

arpabet line found: 1 | pinyin line found: 1
patched
total symbols: 114
dictionary phones: 47 | missing from symbols: []
[64, 88, 64, 106, 66, 104, 74] -> {A DD A V I L OO}


In [19]:
import os
os.chdir("/content/FastSpeech2")
!cat train.py
print("-------------------- utils/model.py (first 90 lines) --------------------")
!sed -n '1,90p' utils/model.py

import argparse
import os

import torch
import yaml
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

from utils.model import get_model, get_vocoder, get_param_num
from utils.tools import to_device, log, synth_one_sample
from model import FastSpeech2Loss
from dataset import Dataset

from evaluate import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def main(args, configs):
    print("Prepare training ...")

    preprocess_config, model_config, train_config = configs

    # Get dataset
    dataset = Dataset(
        "train.txt", preprocess_config, train_config, sort=True, drop_last=True
    )
    batch_size = train_config["optimizer"]["batch_size"]
    group_size = 4  # Set this larger than 1 to enable sorting in Dataset
    assert batch_size * group_size < len(dataset)
    loader = DataLoader(
        dataset,
        batch_size=batch_size * group_size,
        shuff